# Lab 3 : When search plus generate is not enough

*Week 3 · Utrains LLMOps 8-Week Course*

Run each cell from the top. Read the printed output before you run the next cell.


## What we are achieving in this lab

Lab 2 worked because the HR policy was a **clean** file: one topic per section, no copies of the same paragraph, no old version sitting next to a new version.

A company index is usually not clean. The same retrieve-then-generate loop still runs. The documents are messier, so the answers fail in named ways.

This lab does **not** use `hr_policy.txt`. It stores seven short documents that are built to trip search. You will ask a question, **print the chunks that come back**, and name what went wrong.

You will see five failures:

1. An exact product code buried in a paragraph about lunch  meaning search looks at the whole paragraph and can miss the code.
2. Two copies of the same refund sentence  they take two of the three slots you asked for.
3. An old 30-day refund rule and a new 14 day rule, with no filter for which one is in force.
4. An old 2023 outage retrieved for a live "why did payments fail?" question, because the words match.
5. A parental-leave question when parental leave is not in this index. Search still returns three chunks. A vague prompt may invent leave days from the PTO chunk.

You do **not** fix these today. Week 4 is where you fix them. Today you need to recognise them, the way you would in a company review: print the retrieved text, then decide if generate should even run.

**Keys.** Same `.env` as Lab 2: `OPENAI_API_KEY` and `ANTHROPIC_API_KEY`.

**Cost.** A few embedding calls and two short Claude replies. Fractions of a cent.


## Same loop, different documents

Nothing in the code path changes from Lab 2: embed the documents, store them, search, then (in the last failure) generate.

What changes is the **input**. These seven documents are short on purpose so you can read every retrieved row.


### Step 1. Load the API keys

Same as Lab 2. `load_dotenv()` reads `.env` in this folder.


In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads .env from this folder


### Step 2. Store seven messy documents

Each `Document` is one short text plus a label in `metadata` (an `id`, sometimes a `version`). We do not split these: they are already small.

Read the texts in the next cell. You will search them in the five failures below.


In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# Seven short documents. Each one is a failure you will see below.
CORPUS = [
    Document(
        page_content="Error E4221 occurs when the payment service receives a malformed VAT field.",
        metadata={"id": "e4221"},
    ),
    Document(
        page_content=(
            "Lunch this week is tomato soup, grilled cheese, and apple pie. "
            "Label your food. Recycling is by the stairs. "
            "Warehouse item PROD-A412-X3 is a replacement air filter. "
            "Do not leave dishes in the sink."
        ),
        metadata={"id": "buried-sku"},
    ),
    Document(
        page_content="Refund policy: 30 days. Submit a ticket within the window.",
        metadata={"id": "refund-v1-a", "version": "1"},
    ),
    Document(
        page_content="Refund policy: 30 days. Submit a ticket within the window.",
        metadata={"id": "refund-v1-b", "version": "1"},
    ),
    Document(
        page_content="Refund policy v2 (in effect from January 2026): 14 days.",
        metadata={"id": "refund-v2", "version": "2"},
    ),
    Document(
        page_content="Our annual outage in 2023 caused payment failures across the region.",
        metadata={"id": "incident-2023"},
    ),
    Document(
        page_content="PTO is unlimited with a 2-week annual minimum.",
        metadata={"id": "pto"},
    ),
]

vectorstore = InMemoryVectorStore.from_documents(CORPUS, embedding=embeddings)


def show_hits(query: str, k: int = 3) -> list[Document]:
    # Same search as Lab 2. Print every returned chunk so we can name the failure.
    print("Question:", query)
    print()
    hits = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(query)
    for i, hit in enumerate(hits, start=1):
        print(i, "[" + str(hit.metadata.get("id")) + "]", hit.page_content)
    print()
    return hits


print("stored", len(CORPUS), "docs")
print("embed :", EMBED_MODEL)


### Failure 1. An exact code buried in the wrong topic

`PROD-A412-X3` is a warehouse SKU (a product code). It sits inside a paragraph that is mostly about lunch.

Lab 1: **keyword** search looks for the same words. It would find that code. **Cosine** (meaning search) looks at the whole paragraph. The paragraph is about lunch, so the product-code question can miss that row, or rank it low.

Ask for the SKU. Read which rows come back.


In [ ]:
show_hits("What is SKU PROD-A412-X3?")


If `buried-sku` is not the first row, meaning search paid more attention to "lunch" than to the product code. Keyword search would have matched `PROD-A412-X3` on that row.

Week 4's fix is to run **both** (keywords and meaning) and combine the lists. We do not build that today.


### Failure 2. Two copies of the same sentence take two slots

We stored the 30-day refund sentence twice. Search with `k=3` returns three rows. Two of those rows can be the same sentence. Then the newer 14-day rule may not appear in the three.

Ask how long a refund takes. Count how many of the three rows are the 30-day copy.


In [ ]:
show_hits("How long do I have to request a refund?", k=3)


If you see both `refund-v1-a` and `refund-v1-b`, you stored the same rule twice and search paid for both.

Week 4: delete duplicate text **before** you embed.


### Failure 3. Two versions of the rule, no way to keep only the current one

30 days and 14 days are both "about refunds." Search scores meaning. It cannot read the `version` label we stored in metadata unless we add a **filter** (a rule like "only version 2").

Ask for five rows so you see both policies in the list.


In [ ]:
show_hits("How long do I have to request a refund?", k=5)


Both policies appear. When you generate later, the model may pick one, mix them, or hedge. All three are wrong if only the 14-day rule is in force.

Week 4: filter on metadata (version, dates). Search is then "closest in meaning **and** matching this label," not meaning alone.


### Failure 4. The words match, but it is the wrong job

An employee asking "Why did payments fail?" usually means a **live** checkout problem. This index also has a 2023 outage about payment failures. Meaning search will treat them as close.

Read the rows. If `incident-2023` appears, search did what it is designed to do: same topic. The **job** was "current incident," which this index does not know.


In [ ]:
show_hits("Why did payments fail?")


If `incident-2023` is in the list, that is not a bug in cosine. The index has no date filter and no "current vs archive" label.

Week 4: store dates in metadata, or run a second scoring pass (a **reranker**) that reads the question and each chunk together.


### Failure 5. The answer is not in the index

There is no parental-leave document here. Lab 2's HR policy had one. Same question, different files.

Search still returns three chunks because `k=3`. PTO is a related kind of leave, so it often comes back.

Then we generate **twice** with the same chunks:

1. A vague prompt: "You are a helpful HR assistant. Answer the question." That is Week 2's weak system message. It may invent leave days from the PTO chunk.
2. Lab 2's grounded prompt: answer only from the context; if it is not there, say you cannot find it.


In [ ]:
hits = show_hits("What is the company's parental leave policy?")

parts = []
for hit in hits:
    parts.append(hit.page_content)
context = "\n\n---\n\n".join(parts)

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)
question = "What is the company's parental leave policy?"

# Vague: no rule that the answer must come from the chunks.
naive = llm.invoke(
    [
        ("system", "You are a helpful HR assistant. Answer the question."),
        ("user", "Context:\n" + context + "\n\nQuestion: " + question),
    ]
)
# Grounded: same rule as Lab 2.
grounded = llm.invoke(
    [
        (
            "system",
            "You answer using ONLY the provided context. "
            "If the context does not contain the answer, say you cannot find it. Do not guess.",
        ),
        ("user", "Context:\n" + context + "\n\nQuestion: " + question),
    ]
)

print("--- vague prompt ---")
print(naive.content)
print()
print("--- grounded prompt (Lab 2) ---")
print(grounded.content)


The grounded prompt should refuse. The vague prompt may invent parental-leave days from the PTO chunk. That answer can sound like HR and still be false.

A prompt is not a guarantee. Later, production code often adds a score cutoff: if nothing is close enough, **your code** says "I don't know" and does not call the model. You measure that cutoff on your own data (Lab 1 cosine scores).

## What you should be able to explain

You should be able to say these in your own words:

- Lab 2's loop still fails when the documents are messy: a buried product code, duplicate sentences, two versions of a rule, an old incident next to a live question, and a missing fact.
- Search with `k=3` always returns three chunks. Wrong chunks mean a wrong answer, even with a good prompt.
- Today you only name the failures. Week 4 is where you combine keyword and meaning search, remove duplicates, filter on metadata, rerank, and add a score cutoff.

Week 3's question was: can I make a model answer from my company's HR policy? Yes — and this lab is the list of ways that sentence is incomplete until you check the retrieved text.
